# Photo Studio Data Analysis 

Author: Clive Marvelous

Date: 22/02/2026

# 1. Bank Statement Analysis

Initialise - Import all necessary libarary 

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
import supabase
import os 
from dotenv import load_dotenv
import re 

Get data from supabase and store in dataframe

In [ ]:
# load .env to get url and key for supabase client
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

# create supabase client
supabase: Client = supabase.create_client(url, key)




In [ ]:
# get data from supabase and store in pd dataframe
response = supabase.table("transactions").select("*").execute()

transaction_df = pd.DataFrame(response.data)
transaction_df = transaction_df.sort_values(by = 'id').reset_index(drop=True)
print(transaction_df.head())

### 1.1 Descriptive Analysis

- Explore data types and statistical summary to get a broad idea of the dataset

In [ ]:
# convert tanggal to datetime type 
transaction_df['tanggal'] = pd.to_datetime(transaction_df['tanggal'])

# inspect dataframe info
print("######## INFO #########")
print(transaction_df.info())
print()

# Get all unique categories and subcategories
print("######## CATEGORIES #########")
print(transaction_df['category'].unique())
print()
print("######## SUBCATEGORIES #########")
print(transaction_df['subcategory'].unique())

### 1.2 Exploratory Analysis

##### SALDO
- Get saldo values and plot -> See trend 

In [ ]:
# get saldo column 
saldo_df = transaction_df[['tanggal', 'mutasi']]
saldo_df['saldo'] = saldo_df['mutasi'].cumsum()

# convert saldo to millios 
saldo_df["saldo"] = saldo_df["saldo"] / 1e6
print(saldo_df.head())

# get numeric dates values 
saldo_df['tanggal_numeric'] = mdates.date2num(transaction_df['tanggal'])

# get linear regression coefficients and predicted values
results = sm.OLS.from_formula('saldo ~ tanggal_numeric', data=saldo_df).fit()
intercept, slope = results.params
fitted_values = results.predict(saldo_df['tanggal_numeric'])

# print min max of saldo 
print()
print("######## START & END OF SALDO #########")
tanggal_awal = saldo_df['tanggal'].min()
tanggal_akhir = saldo_df['tanggal'].max()
print(f"Start date: {tanggal_awal.date()} with saldo of Rp. {saldo_df.loc[saldo_df['tanggal'] == tanggal_awal, 'saldo'].values[0]:.2f} Jt")
print(f"End date: {tanggal_akhir.date()} with saldo of Rp. {saldo_df.loc[saldo_df['tanggal'] == tanggal_akhir, 'saldo'].values[-1]:.2f} Jt")

# print regression coefficients
print()
print("######## REGRESSION COEFFICIENTS OF SALDO #########")
print(results.params)

# plot saldo over tanggal 
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(saldo_df["tanggal"], saldo_df["saldo"])
ax.plot(saldo_df["tanggal"], fitted_values, color="red", linestyle="--")

ax.set_xlabel("Date")
ax.set_ylabel("Cash")
ax.set_title("Cash in Bank Over Time")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

INSIGHT: Trend - Cumulative revenue is increasing over time at Rp. 20,659 per day (linear regression)

- Explore the validatity of linear regression assumptions (linearity, normality of residuals, homoscedasticity)

In [ ]:
# check if linear regression assumptions are met
# 1. normally distributed residuals 
residuals = results.resid
sns.histplot(residuals, kde=True)
plt.title("Distribution of Residuals")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.show()
plt.clf()

# 2. homoscedasticity
sns.scatterplot(x=fitted_values, y=residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Fitted Values")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()
plt.clf()

INSIGHT: 
1. From plot, saldo growth can be assumed to be linear 
2. Residuals (Actual values - predicted values) are generally normally distributed 
3. Homoscedasticity is not randomly distributed, instead it suggests non-linear behaviour (cyclical)

Conclusion: Linear regression may not be the best model to predict saldo growth


##### INCOME & EXPENSES

- Create revenue dataframes and plot -> Visualise trend 

In [ ]:
# revenue dataframes
revenue_df = transaction_df[transaction_df['category'] == 'REVENUE']
   
# drop SALDO AWAL row for better accuracy from revenue df 
revenue_df = revenue_df.drop(index=0)

# convert mutasi to millions 
revenue_df["mutasi"] = revenue_df["mutasi"] / 1e6

# get TOTAL monthly revenues 
monthly_revenue_df = revenue_df.groupby(revenue_df['tanggal'].dt.to_period('M'))['mutasi'].sum().reset_index()
monthly_revenue_df = monthly_revenue_df.set_index('tanggal')
print(monthly_revenue_df.head())

# get monthly revenue by SUBCATEGORY
monthly_revenue_by_subcategory_df = revenue_df.groupby([revenue_df['tanggal'].dt.to_period('M'), 'subcategory'])['mutasi'].sum().reset_index()

# pivot the df 
monthly_revenue_pivot = monthly_revenue_by_subcategory_df.pivot(index='tanggal', columns='subcategory', values='mutasi').fillna(0)
print(monthly_revenue_pivot.head())

# bar plot of revenue by month
monthly_revenue_df.plot(
    kind='bar',
    stacked=False,
    figsize=(10, 6),
    color="#62D358",
    legend=''
)

avg_line = plt.axhline(
    y=monthly_revenue_df['mutasi'].mean(),
    color="red",
    linestyle="--",
    label="Average Monthly Income"
)

plt.xlabel("Month")
plt.ylabel("Income")
plt.title("Monthly Income")
plt.xticks(rotation=45)
plt.legend(handles=[avg_line])  # only show the axhline entry
plt.tight_layout()
plt.show()



In [ ]:
# get revenue for January 2025 and January 2026
revenue_jan25 = monthly_revenue_df.loc['2025-01', 'mutasi']
revenue_jan26 = monthly_revenue_df.loc['2026-01', 'mutasi']

# calculate percentage change from Jan 2025 to Jan 2026
percentage_change_jan = (revenue_jan26 - revenue_jan25) / revenue_jan25 * 100

print(f'Percentage change from Jan 2025 to Jan 2026: {percentage_change_jan:.2f}%')

INSIGHT: Revenue fluctuates -> Might be cyclical (require more data to prove) 

In [ ]:
# get all necessary data from revenue df
print(monthly_revenue_df.describe(include='all'))

# get best and worst performing months in terms of revenue
best_month = monthly_revenue_df['mutasi'].idxmax()
worst_month = monthly_revenue_df['mutasi'].idxmin()

print()
print(f"Best performing month: {best_month} with revenue of {monthly_revenue_df.loc[best_month, 'mutasi']:.2f} Jt")
print(f"Worst performing month: {worst_month} with revenue of {monthly_revenue_df.loc[worst_month, 'mutasi']:.2f} Jt")

- Expense dataframe and plot -> Trend 
1. Bar plot of total expenses and breakdown of expense subcategories

In [ ]:

expense_df = transaction_df[transaction_df['category'] != 'REVENUE']
expense_df["mutasi"] = expense_df["mutasi"] / 1e6
monthly_expense_df = expense_df.groupby(expense_df['tanggal'].dt.to_period('M'))['mutasi'].sum().reset_index()


# expense dataframe
expense_df = transaction_df[transaction_df['category'] != 'REVENUE']

# convert mutasi to millions 
expense_df["mutasi"] = expense_df["mutasi"] / 1e6

# get TOTAL monthly expenses 
monthly_expense_df = expense_df.groupby(expense_df['tanggal'].dt.to_period('M'))['mutasi'].sum().reset_index()
monthly_expense_df = monthly_expense_df.set_index('tanggal')
# print(monthly_expense_df.head())

# get monthly expense by CATEGORY
monthly_expense_by_category_df = expense_df.groupby([expense_df['tanggal'].dt.to_period('M'), 'category'])['mutasi'].sum().reset_index()

# pivot the df 
monthly_expense_by_category_pivot = monthly_expense_by_category_df.pivot(index='tanggal', columns='category', values='mutasi').fillna(0)
monthly_expense_by_category_pivot['OTHERS'] = monthly_expense_by_category_pivot.FINANCIAL + monthly_expense_by_category_pivot.TAX
monthly_expense_by_category_pivot = monthly_expense_by_category_pivot.drop(columns=['FINANCIAL', 'TAX']).abs()
print(monthly_expense_by_category_pivot.head())

# print all expense categories 
print("######## EXPENSE CATEGORIES #########")
print(monthly_expense_by_category_pivot.columns)

# rearrange columns 
desired_order = ['PERSONNEL', 'OPERATIONS', 'MARKETING', 'DIRECT', 'EQUIPMENT', 'OTHERS']
monthly_expense_by_category_pivot = monthly_expense_by_category_pivot[desired_order]

# bar plot color palette for expense categories
palette = [
    "#1E4B66",  # Organic Search (dark blue-teal)
    "#62689E",  # Direct (muted indigo)
    "#C990B9",  # Social (soft purple-pink)
    "#EC7979",  # Referral (soft red)
    "#F0A532",   # Other (warm orange)
    "#523100"   # Other (warm orange)
]

monthly_expense_by_category_pivot.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=palette,
    legend=True
)

plt.xlabel("Month")
plt.ylabel("Expense")
plt.title("Expense by Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# get all statistics about expenses 
print('###### Expenses Statistics ######')
print(monthly_expense_df.describe(include='all'))

# Monthly fixed cost 
fixed_cost = monthly_expense_by_category_df[monthly_expense_by_category_df['category'].isin(['PERSONNEL', 'OPERATIONS'])]['mutasi'].sum()
avg_monthly_fixed_cost = fixed_cost / monthly_expense_by_category_df['tanggal'].nunique()

print()
print("######## FIXED COST #########")
print(f"Total fixed cost: Rp. {fixed_cost:.2f} Jt")
print(f"Monthly avg. fixed cost: Rp. {avg_monthly_fixed_cost:.2f} Jt")

INSIGHT: 
1. Fixed costs (Personnel & Operations) are approximately the same every month (Assuming that electricity is considered as fixed, for now)

- Pie chart of expenses

In [ ]:
# sum all expenses by category 
total_expense_by_category = monthly_expense_by_category_pivot.sum(axis=0).reset_index()
total_expense_by_category.rename(columns={0: 'total_expense'}, inplace=True)

# sort by descending total expense
total_expense_by_category = total_expense_by_category.sort_values('total_expense', ascending=False)
print(total_expense_by_category)


# create pie chart of percentage of each category
fig, ax = plt.subplots(figsize=(10, 6))

def autopct_func(pct):
    return f"{pct:.1f}%" if pct >= 3 else ""

wedges, texts, autotexts = ax.pie(
    total_expense_by_category["total_expense"],
    labels=None,
    autopct=autopct_func,
    startangle=90,
    colors=palette,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5, "width": 0.8},
    pctdistance=0.72,
    radius=1.2
)

for t in autotexts:
    t.set_fontsize(14)
    t.set_color("white")

title = ax.set_title("Avg. Expense by Category", fontsize=25)
title.set_fontweight('bold')

legend = ax.legend(
    wedges,
    total_expense_by_category["category"],
    title="CATEGORY",
    title_fontsize=12,
    fontsize=10,
    loc="lower left",
    #bbox_to_anchor=(-0.1, 0),
    handlelength=2,
    framealpha=0.95
    
)

legend.get_title().set_fontweight('bold')

plt.tight_layout()
plt.show()

- Side by side plot of revenue and expense

In [ ]:
x = np.arange(len(monthly_expense_by_category_pivot.index))
width = 0.4

fig, ax = plt.subplots(figsize=(12,6))

# --- stacked expense bars ---
bottom = np.zeros(len(monthly_expense_by_category_pivot.index))

for i, col in enumerate(monthly_expense_by_category_pivot.columns):
    ax.bar(
        x + width/2,
        monthly_expense_by_category_pivot[col],
        width,
        bottom=bottom,
        label=col,
        color=palette[i]
    )
    bottom += monthly_expense_by_category_pivot[col]


# --- income bars ---
ax.bar(
    x - width/2,
    monthly_revenue_df['mutasi'],
    width,
    label="REVENUE",
    color="#62D358"
)

ax.set_xticks(x)
ax.set_xticklabels(monthly_expense_by_category_pivot.index.astype(str), rotation=45)

ax.set_xlabel("Month")
ax.set_ylabel("Amount")
ax.set_title("Monthly Income & Expenses Side-by-side")

ax.legend(loc="center left", bbox_to_anchor=(1,0.7))

plt.tight_layout()
plt.show()


INSIGHT: The plot suggests that revenue tends to be higher when there is marketing spending the previous month  

- Find connection between revenue and marketing  spending

In [ ]:
# print average marketing spending
print('###### Average Marketing Spending ######')
average_marketing_spending = monthly_expense_by_category_pivot['MARKETING']
print(average_marketing_spending.describe(include='all'))

# Get average revenue from Feb 25 to Aug 25
avg_revenue_feb25_aug25 = monthly_revenue_df.loc['2025-02':'2025-08', 'mutasi'].mean()

# Get average revenue from Sep 25 to Jan 26
avg_revenue_sep25_jan26 = monthly_revenue_df.loc['2025-09':'2026-01', 'mutasi'].mean()

# print the average revenues
print()
print(f"Average revenue from Feb 25 to Aug 25: Rp. {avg_revenue_feb25_aug25:.2f} Jt")
print(f"Average revenue from Sep 25 to Jan 26: Rp. {avg_revenue_sep25_jan26:.2f} Jt")



INSIGHT: Average revenue when there is marketing spending is higher than when there is no marketing spending. However, is spending on marketing (avg 1.3m) per month is worth it as it only increases revenue average by approx. 0.9m monthly?

##### NET CASHFLOW 
- Plot net cashflow

In [ ]:
# monthly net cashflow dataframe 
net_cashflow_df = monthly_revenue_df.copy()
net_cashflow_df['mutasi'] = monthly_revenue_df['mutasi'] + monthly_expense_df['mutasi']
print(net_cashflow_df.head())

# assign color based on positive or negative net cashflow
net_cashflow_df['sign'] = net_cashflow_df['mutasi'].apply(lambda x: 'POSITIVE' if x >= 0 else 'NEGATIVE')
cashflow_palette = {'POSITIVE': "#62D358", 'NEGATIVE': "#F07070"}

# plot net cashflow over time
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=net_cashflow_df,
    x="tanggal",
    y="mutasi",
    hue="sign",
    palette=cashflow_palette,
    dodge=False,
    legend=False
)

plt.xlabel("Month")
plt.ylabel("Total Mutas (Jt)")
plt.title("Monthly Net Cashflow")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

- Trend and statistics

In [ ]:
# print all statistics about net cashflow
print('###### Net Cashflow Statistics ######')
print(net_cashflow_df['mutasi'].describe(include='all'))

# best and worst performing months in terms of net cashflow
best_cashflow_month = net_cashflow_df['mutasi'].idxmax()
worst_cashflow_month = net_cashflow_df['mutasi'].idxmin()
print()
print('###### Best & Worst Performing Months in Terms of Net Cashflow ######')
print(f"Best performing month: {best_cashflow_month} with net cashflow of Rp. {net_cashflow_df.loc[best_cashflow_month, 'mutasi']:.2f} Jt")
print(f"Worst performing month: {worst_cashflow_month} with net cashflow of Rp. {net_cashflow_df.loc[worst_cashflow_month, 'mutasi']:.2f} Jt")

# cashflow sum and monthly average 
total_cashflow = net_cashflow_df['mutasi'].sum()
avg_monthly_cashflow = total_cashflow / len(net_cashflow_df)
print()
print('###### Total & Average Monthly Net Cashflow ######')
print(f"Total net cashflow: Rp. {total_cashflow:.2f} Jt")
print(f"Average monthly net cashflow: Rp. {avg_monthly_cashflow:.2f} Jt")

# get std of positive and negative cashflow months
positive_cashflow_std = net_cashflow_df[net_cashflow_df['mutasi'] >= 0]['mutasi'].std()
negative_cashflow_std = net_cashflow_df[net_cashflow_df['mutasi'] < 0]['mutasi'].std()
print()
print('###### Standard Deviation of Positive & Negative Cashflow Months ######')
print(f"Standard deviation of positive cashflow months: Rp. {positive_cashflow_std:.2f} Jt")
print(f"Standard deviation of negative cashflow months: Rp. {negative_cashflow_std:.2f} Jt")

INSIGHT: 
1. Positive average monthly cashflow -> Growing 
2. Average net cashflow only 0.31 and total of 4.03 for a year -> Very slow and stagnant growth
3. Good months: May, August, December. Bad months: January, February, June.
4. Fluctuating monthly cashflow -> Suggests might be cyclical (need more data for evidence)
5. From (3) -> Goal: Minimise losses during bad months and maximise profits during good months
6. Std of positive cashflow is relatively big compared to the negative -> Negative cashflow is more stable than the positive cashflow -> Need to minimise positive cashflow fluctuations



##### Growth Projections and Target 

In [ ]:

# create future dates
future_dates = pd.date_range(start=saldo_df['tanggal'].max(), end='2026-12-31', freq='ME')
future_dates_numeric = mdates.date2num(future_dates)

# concatenate future dates 
future_tanggal = np.concatenate((saldo_df['tanggal'], future_dates))
future_tanggal_numeric = np.concatenate((saldo_df['tanggal_numeric'], future_dates_numeric))

# get linear regression of predicted values -> GROWTH PROJECTION
future_df = pd.DataFrame({
    "tanggal_numeric": future_tanggal_numeric
})
projection_fitted_values = results.predict(future_df)

final_predicted_saldo = projection_fitted_values.iloc[-1]

# exponential growth projection with target of 200 Jt by end of 2026 -> TARGET PROJECTION
# define starting and target value in millions
target_value = 200 
start_value = saldo_df['saldo'].iloc[-1]

# define start and end dates 
start_date = future_dates_numeric[0]
end_date = future_dates_numeric[-1]

# get period
T = end_date - start_date

# calculate growth rate
growth_rate = np.log(target_value / start_value) / T

# generate exponential curve 
future_target = start_value * np.exp(growth_rate * (future_dates_numeric - start_date))

# plot saldo over tanggal
fig, ax = plt.subplots(figsize=(10, 6))

# plot saldo
ax.plot(saldo_df['tanggal'], saldo_df["saldo"])
# plot linear regression -> GROWTH PROJECTION
ax.plot(future_tanggal, 
        projection_fitted_values, 
        color="red", 
        linestyle="--", 
        label="Growth Projection (Linear)"
)
# plot exponential regression -> TARGET PROJECTION
ax.plot(
    future_dates,
    future_target,
    linestyle=":",
    linewidth=2,
    label="Target (Exponential)"
)

ax.set_xlabel("Date")
ax.set_ylabel("Saldo (Jt)")
ax.set_title("Saldo Growth Projection")

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# print the growth rate
print(f"Target growth rate: {growth_rate*100:.2f}% per day or {growth_rate*30*100:.2f}% per month")

# print amount of saldo by deadline 
print(f"By 2027-12-31, total saldo: Rp. {final_predicted_saldo:.2f} Jt")


INSIGHT: 
1. There has to be a drastic change in business strategy to bring in more revenue because if we follow current business model and strategy, by the end of the year, we will only bring in a cumulative revenue of Rp 44.29Jt. 
2. Ambitious target: Rp. 200Jt by the end of the year

# 2. Booking Spreadsheet

### 2.1 Descriptive Analysis

- Import, clean and restructure data 

In [ ]:
# booking spreadsheet filepath
bookings_filepath = r""

# import to dataframe and inspect
bookings_df = pd.read_csv(bookings_filepath)

# rename columns 
bookings_df.rename(columns={'No.': 'no',
                            'Time': 'time',
                            'Full Name': 'full_name',
                            'NOTE': 'note',
                            'EMAIL': 'email',
                            'WhatsApp': 'whatsapp',
                            'CODE': 'booking_id',
                            'Date': 'date',
                            'Payment': 'paid'}, 
                   inplace=True)


# clean dataframe 
# 1. delete all rows with NaN time 
bookings_df = bookings_df.dropna(subset=['time'])

# 2. replace no with unique ascending order values and change type to int
bookings_df['no'] = range(1, len(bookings_df) + 1)
bookings_df['no'] = bookings_df['no'].astype(int)

# 3. change time to datetime.time type
bookings_df["time"] = bookings_df["time"].astype(str).str.strip().str.slice(0, 5)
bookings_df["time"] = pd.to_datetime(bookings_df["time"], format="%H:%M", errors="coerce").dt.time

# 4. change full_name of all rows with note that contains 'AGENCY'
# identify rows where note contains 'AGEN'
mask = bookings_df['note'].str.contains('AGEN', case=False, na=False)

# swap full_name and note for those rows
bookings_df.loc[mask, ['note']] = bookings_df.loc[mask, ['full_name']].values

# replace full_name to 'AGENCY'
bookings_df.loc[mask, ['full_name']] = 'AGENCY'

# 5. clean full_name into three separate columns -> full_name (all capslock), background_color, extra 
bookings_df[['full_name', 'background_color', 'extra']] = (bookings_df['full_name']
                                                           .str.upper()
                                                           .str.extract(r'^\s*(.+?)\s*\(\s*(.+?)\s*\)\s*(\S+)?\s*$')
)

# 6. clean the date column -> remove weekday names and convert to datetime 
# capslock the date 
bookings_df['date'] = bookings_df['date'].str.title()

# map bulan to MM
bulan = {
    'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}
# convert date to datetime obj
bookings_df['date'] = (
    bookings_df['date']
    .str.replace(r'^\S+[,\s]\s*', '', regex=True)
    .str.replace('|'.join(bulan.keys()), lambda m: bulan[m.group()], regex=True)
    .str.replace(r'\s+', ' ', regex=True)   # fix double spaces
    .str.strip()
    .pipe(lambda s: pd.to_datetime(s, format='%d %m %Y'))
)


# 7. clean note column -> create 'discount' and 'booking' columns 
# default booking = 1, set 0 if note contains 'NON BOOKING'
bookings_df['booking'] = 1
mask_non_booking = bookings_df['note'].str.contains('NON BOOKING', case=False, na=False)
bookings_df.loc[mask_non_booking, 'booking'] = 0

# extract discount percentage into new column
bookings_df['discount'] = (
    bookings_df['note']
    .str.extract(r'(?i)DISC\s+(\d+)%')
    .astype(float)
    .fillna(0)
)

# clean note: remove 'NON BOOKING' and 'DISC X%' and leftover '+'
bookings_df['note'] = (
    bookings_df['note']
    .str.replace(r'NON BOOKING', '', case=False, regex=True)
    .str.replace(r'DISC\s+\d+%', '', case=False, regex=True)
    .str.replace(r'\+\s*\+', '+', regex=True)   # remove double +
    .str.strip(r'+\s ')                         # remove leading/trailing +
    .str.strip()
)

# 8. convert paid column to int -> either 0 or 1 
bookings_df['paid'] = bookings_df['paid'].astype(int)

# 9. rearrange column order 
booking_column_order = ['no', 'date', 'time', 'full_name', 'background_color', 'booking', 'discount', 'note', 'extra', 'paid', 'email', 'whatsapp']
bookings_df = bookings_df[booking_column_order]

# 10. remove invalid rows -> full_name is nan, note is nan, paid is 0
mask_remove = bookings_df['full_name'].isna() & (bookings_df['paid'] == 0) & bookings_df['note'].isna()
bookings_df = bookings_df[~mask_remove].reset_index(drop=True)

# 11. set full_name to either AGENCY or NO NAME for current NaN full_name 
bookings_df.loc[[101, 480, 632], 'full_name'] = 'NO NAME'
mask = bookings_df['full_name'].isna()
bookings_df.loc[mask, 'full_name'] = 'AGENCY'

# 12. clean WA number -> remove - and 0 
bookings_df['whatsapp'] = (
    bookings_df['whatsapp']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)  # remove dashes
    .str.lstrip('0')                     # remove leading 0
)

# 13. remove invalide WA number
bookings_df['whatsapp'] = bookings_df['whatsapp'].where(
    bookings_df['whatsapp'].str.match(r'^8[1-9]\d{7,10}$'), other=None
)





In [ ]:
# import database from supabase
# booking_response = supabase.table("bookings").select("*").execute()

# bookings_df = pd.DataFrame(booking_response.data)

# # change datatype
# bookings_df['date'] = pd.to_datetime(bookings_df['date'])
# bookings_df['time'] = pd.to_datetime(bookings_df['time'], format='%H:%M:%S').dt.time


In [ ]:
# inspect
print(bookings_df.head(15))
print(bookings_df.info())

### 2.2 Exploratory Analysis
- booking statistics 

In [ ]:
bookings_df.describe(include='all')


##### Bookings 
- Booking percentage 

In [ ]:
# total percentage of bookings and non-booking 
print('######## BOOKING v. NON-BOOKING PERCENTAGE #########')
booking_percentage = bookings_df.booking.mean() * 100
print(f'Booking: {booking_percentage:.2f} % \nNon-booking: {100 - booking_percentage:.2f} %')



- Monthly booking (online and non-booking) 

In [ ]:
# monthly booking 
monthly_booking_count_df = (
    bookings_df.groupby(bookings_df['date'].dt.to_period('M'))
    .size()
    .reset_index()
    .set_index('date')
)

# rename column
monthly_booking_count_df.rename(columns={0: 'booking_count'}, inplace=True)

# print total monthly booking 
print('######### MONTHLY BOOKING COUNT ###########')
print(monthly_booking_count_df)

# bar plot 
monthly_booking_count_df.plot(
    kind='bar', 
    figsize=(10, 6),
    color="#AB51F5",
    legend=False
)

plt.xlabel("Month")
plt.ylabel("Booking Count")
plt.title("Monthly Booking Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

INSIGHT: 
1. Booking pattern is similar to revenue -> Since most of our revenue is generated from bookings 
2. Most of bookings are made online -> Good as this is more efficient and easier to track and record 
3. Low non-booking suggests that people rarely shows up to the studio 

##### Discount
- Compare booking count for discount

In [ ]:
# df of total booking counts by discount 
total_bookings_by_disc_df = (
    bookings_df.groupby('discount')
    .size()
    .reset_index()
    .rename(columns={0: 'booking_count'})
)

# df of monthly booking counts by discount 
monthly_bookings_by_disc_df = (
    bookings_df.groupby([bookings_df.date.dt.to_period('M'), 'discount'])
    .size()
    .reset_index()
    .rename(columns={0: 'booking_count'})
)
# pivot monthly df 
monthly_bookings_by_disc_pivot = monthly_bookings_by_disc_df.pivot(index='date', columns='discount', values='booking_count').fillna(0)
monthly_bookings_by_disc_pivot = monthly_bookings_by_disc_pivot.astype(int)

# print
print('######### TOTAL BOOKING COUNT BY DISCOUNT ########')
print(total_bookings_by_disc_df)
print()


# focusing on 0%, 25%, 30% discount, bar plot 
monthly_bookings_by_disc_pivot = monthly_bookings_by_disc_pivot.drop(columns=[20, 50])

# print('######### MONTHLY BOOKING COUNTS BY DISCOUNT #########')
# print(monthly_bookings_by_disc_pivot)
# print()
print('######### MONTHLY BOOKING COUNTS BY DISCOUNT STATISTICS #########')
print(monthly_bookings_by_disc_pivot.describe(include='all'))

# percentage of booking with discount and without discount 
disc_0_count = total_bookings_by_disc_df.loc[total_bookings_by_disc_df['discount'] == 0, 'booking_count'].values[0]
total_count = total_bookings_by_disc_df['booking_count'].sum()
disc_0_percentage = disc_0_count / total_count * 100

print()
print(f"Percentage of discount 0: {disc_0_percentage:.2f} % out of {total_count:.0f} bookings")



INSIGHT: 
1. On avg, 29 non-discounted bookings a month with std 10 -> stable but quite stagnant demand 
2. More discount results in more booking on average -> but for an increase of 2.5 bookings a month, is it worth increasing the discount by 5%? 

In [ ]:
# bar plot 
monthly_bookings_by_disc_pivot.drop(columns=[0]).plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=["#F89C32", "#F85033"]
)
plt.xlabel("Month")
plt.ylabel("Booking Count")
plt.title("Monthly Booking Count by Discount")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

INSIGHT: 
1. Assuming that non-discounted booking remains flat, discount does increase demand -> Need further evidence to how much and is it worth it since discounts are available everyday
2. From plot, 25% or 30% discount provide a similar boost to demand 


##### Weekday Analysis 
- Number of bookings for different day 

In [ ]:
# weekday column 
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
bookings_df['weekday'] = pd.Categorical(bookings_df['date'].dt.day_name(), categories=weekday_order, ordered=True)

# booking count by weekday 
booking_by_weekday = (
    bookings_df.groupby('weekday')
    .size()
    .reset_index()
    .rename(columns={0: 'booking_count'})
    .set_index('weekday')
)

# print 
print('######## BOOKING COUNT BY WEEKDAY ########')
print(booking_by_weekday)
print()
print(f'Total Count: {booking_by_weekday.booking_count.sum():.0f}')

# plot 
booking_by_weekday.plot(
    kind='bar',
    figsize=(10, 6), 
    color="#4DC3FA",
    legend=False
)
plt.xlabel("Day")
plt.ylabel("Booking Count")
plt.title("Daily Booking Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

INSIGHT: From plot, Saturday and Sunday has the highest booking counts 

##### Time of Day 
- Booking counts for each hour of the day 

In [ ]:
# booking count by time of day 
bookings_df['hour'] = pd.to_datetime(bookings_df['time'].astype(str)).dt.hour

booking_by_time = (
    bookings_df.groupby('hour')
    .size()
    .reset_index(name='booking_count')
    .set_index('hour')
)

# print
print('######## BOOKING COUNT FOR EACH HOUR #######')
print(booking_by_time)

# plot 
booking_by_time.plot(
    kind='bar',
    figsize=(10, 6), 
    color="#4DC3FA",
    legend=False
)
plt.xlabel("Hour")
plt.ylabel("Booking Count")
plt.title("Hourly Booking Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Booking count for each hour (only for saturday & sunday)
booking_by_time_day = (
    bookings_df.groupby(['hour', 'weekday'])
    .size()
    .reset_index(name='booking_count')
    .set_index('hour')
)

# pivot df and keep on weekends 
weekend_days = ['Saturday', 'Sunday']
booking_by_time_weekend_pivot = (
    booking_by_time_day[booking_by_time_day['weekday'].isin(weekend_days)]
    .pivot(columns='weekday', values='booking_count')
    .fillna(0)
    .astype(int)
)

# print
print('######## BOOKING COUNT FOR EACH HOUR (SATURDAY & SUNDAY) #######')
print(booking_by_time_weekend_pivot)

# plot 
booking_by_time_weekend_pivot.plot(
    kind='bar',
    figsize=(10, 6), 
    color=["#4DC3FA", "#EE7171"],
)
plt.xlabel("Hour")
plt.ylabel("Booking Count")
plt.title("Hourly Booking Count (Sat & Sun)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

INSIGHT: 
1. On saturday, booking is cyclical -> Peak at hour 14-15 before starting to slowly drop 
2. On sunday, booking starts of strong (peaking at hour 11) but slowly decline throughout the day (except at hour 13)

In [ ]:
# pivot df and keep on weekends 
weekend_days = ['Saturday', 'Sunday']
booking_by_time_weekday_pivot = (
    booking_by_time_day[~booking_by_time_day['weekday'].isin(weekend_days)]
    .pivot(columns='weekday', values='booking_count')
    .fillna(0)
    .astype(int)
)

# print
print('######## BOOKING COUNT FOR EACH HOUR (WEEKDAYS) #######')
print(booking_by_time_weekday_pivot)

# plot 
booking_by_time_weekday_pivot.plot(
    kind='bar',
    figsize=(10, 6), 
    color=["#4DC3FA", "#EE7171", "#5BFA4D", "#D971EE"],
)
plt.xlabel("Hour")
plt.ylabel("Booking Count")
plt.title("Hourly Booking Count (Weekday)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

##### Repeat Customers 
- Pie chart of customers with 1, 2-3 and >3 bookings 

In [ ]:

# Change names of customer with the same email 
bookings_df.loc[bookings_df['email'] == 'john.doe@gmail.com', 'full_name'] = 'JOHN DOE'
bookings_df.loc[bookings_df['email'] == 'jane.do@gmail.com', 'full_name'] = 'JANE DOE F'



In [ ]:
# count bookings per customer by EMAIL 
bookings_per_customer_email = bookings_df.groupby('email').size()

# categorize customers
def categorize(n):
    if n == 1:
        return '1'
    elif n <= 3:
        return '2-3'
    else:
        return '>3'

customer_frequency_df = (
    bookings_per_customer_email
    .apply(categorize)
    .value_counts()
    .reset_index(name='customer_count')
    .rename(columns={'index': 'booking_frequency'})
    .set_index('booking_frequency')
)

# print
print('######## REPEAT CUSTOMER COUNT BY EMAIL ##########')
print(customer_frequency_df)
print('Reminder: Not all customers provide email address')


# create pie chart of percentage of each category
fig, ax = plt.subplots(figsize=(10, 6))

wedges, texts, autotexts = ax.pie(
    customer_frequency_df["customer_count"],
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    colors=["#3269CF", "#EE7171", "#5BFA4D"],
    wedgeprops={"edgecolor": "white", "linewidth": 1.5, "width": 0.8},
    pctdistance=0.72,
    radius=1.2
)

for t in autotexts:
    t.set_fontsize(14)
    t.set_color("black")

title = ax.set_title("Repeat Customer (Analysis by Email)", fontsize=22)
title.set_fontweight('bold')

legend = ax.legend(
    wedges,
    customer_frequency_df.index,
    title="Frequency",
    title_fontsize=12,
    fontsize=10,
    loc="lower left",
    bbox_to_anchor=(-0.1, 0),
    handlelength=2,
    framealpha=0.95
    
)

legend.get_title().set_fontweight('bold')

plt.tight_layout()
plt.show()

In [ ]:
# print value count of each unique WA number -> inspect anomalies 
print('########## VALUE COUNT OF UNIQUE WA ###########')
print(bookings_df.whatsapp.value_counts().head(10))

# count bookings per customer by EMAIL 
bookings_per_customer_wa = bookings_df.groupby('whatsapp').size()

# categorize customers
customer_frequency_df = (
    bookings_per_customer_wa
    .apply(categorize)
    .value_counts()
    .reset_index(name='customer_count')
    .rename(columns={'index': 'booking_frequency'})
    .set_index('booking_frequency')
)

# print
print()
print('######## REPEAT CUSTOMER COUNT BY WA ##########')
print(customer_frequency_df)
print('\nReminder: Not all customers provide correct WA number')


# create pie chart of percentage of each category
fig, ax = plt.subplots(figsize=(10, 6))

wedges, texts, autotexts = ax.pie(
    customer_frequency_df["customer_count"],
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    colors=["#3269CF", "#EE7171", "#5BFA4D"],
    wedgeprops={"edgecolor": "white", "linewidth": 1.5, "width": 0.8},
    pctdistance=0.72,
    radius=1.2
)

for t in autotexts:
    t.set_fontsize(14)
    t.set_color("black")

title = ax.set_title("Repeat Customer", fontsize=22)
title.set_fontweight('bold')

legend = ax.legend(
    wedges,
    customer_frequency_df.index,
    title="Frequency",
    title_fontsize=12,
    fontsize=10,
    loc="lower left",
    bbox_to_anchor=(-0.1, 0),
    handlelength=2,
    framealpha=0.95
    
)

legend.get_title().set_fontweight('bold')

plt.tight_layout()
plt.show()

INSIGHT: 
1. Analysis of repeat customer using WA and email shows a similar result -> For report, just use the average % of both analysis 
2. Majority of customer is first and last customer 
3. Need to increase % of repeat customer -> Customer acquisition is more expensive than customer retention 

In [ ]:
# final inspection of bookings_df 
print(bookings_df.head(10))
print(bookings_df.info())

# drop weekday and hour columns
bookings_export_df = bookings_df.drop(columns=['weekday', 'hour'])

# save to csv
csv_filepath = r"bookings_export_df.to_csv(csv_filepath, index=False)


